# Multi-Heat Source and PV Bounds Analysis

This notebook evaluates the integration of multiple heat sources, specifically focusing on the impact of PV heat weight bounds and solar thermal lag on the overall thermal model.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta, timezone

from src.analysis.data_loader import DataLoader

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading

Load data related to solar irradiance, PV yield, and multi-source heat contributions.

In [ ]:
loader = DataLoader()
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=14)

features = [
    'indoor_temperature',
    'outdoor_temperature',
    'solar_irradiance',
    'pv_yield',
    'primary_heat_source_power',
    'secondary_heat_source_power',
    'pv_heat_weight'
]

try:
    df = loader.fetch_training_data(start_time=start_time, end_time=end_time, features=features)
    print(f"Loaded {len(df)} records from {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Create dummy data for demonstration
    dates = pd.date_range(start=start_time, end=end_time, freq='1h')
    df = pd.DataFrame(index=dates)
    
    # Simulate diurnal cycle for solar
    hour_of_day = df.index.hour
    solar_profile = np.where((hour_of_day > 6) & (hour_of_day < 18), 
                             np.sin((hour_of_day - 6) * np.pi / 12) * 800, 0)
    
    df['solar_irradiance'] = solar_profile + np.random.normal(0, 50, len(dates))
    df['solar_irradiance'] = df['solar_irradiance'].clip(lower=0)
    
    df['pv_yield'] = df['solar_irradiance'] * 0.15 + np.random.normal(0, 10, len(dates))
    df['pv_yield'] = df['pv_yield'].clip(lower=0)
    
    df['indoor_temperature'] = 20 + np.sin(np.linspace(0, 20, len(dates))) + np.random.normal(0, 0.2, len(dates))
    df['outdoor_temperature'] = 10 + np.sin(np.linspace(0, 20, len(dates))) * 5 + np.random.normal(0, 1, len(dates))
    
    df['primary_heat_source_power'] = np.where(df['indoor_temperature'] < 20.5, 2000, 0)
    df['secondary_heat_source_power'] = df['pv_yield'] * 0.8  # Simplified secondary source
    
    df['pv_heat_weight'] = np.clip(0.5 + np.random.normal(0, 0.1, len(dates)), 0.1, 0.9)
    
    print("Using generated dummy data for demonstration.")

## 2. Solar Thermal Lag Analysis

Analyze the delay between peak solar irradiance and its effect on indoor temperature.

In [ ]:
if 'solar_irradiance' in df.columns and 'indoor_temperature' in df.columns:
    # Select a few sunny days for clearer visualization
    # Assuming dummy data has a clear diurnal cycle
    sample_df = df.iloc[24:96]  # 3 days
    
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    color = 'tab:orange'
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Solar Irradiance (W/m²)', color=color)
    ax1.plot(sample_df.index, sample_df['solar_irradiance'], color=color, label='Solar Irradiance')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.legend(loc='upper left')
    
    ax2 = ax1.twinx()  
    color = 'tab:red'
    ax2.set_ylabel('Indoor Temperature (°C)', color=color)  
    ax2.plot(sample_df.index, sample_df['indoor_temperature'], color=color, label='Indoor Temp')
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.legend(loc='upper right')
    
    fig.tight_layout()  
    plt.title('Solar Irradiance vs Indoor Temperature (Thermal Lag)')
    plt.show()
    
    # Cross-correlation to estimate lag
    # Note: This is a simplified approach. Real analysis might require detrending.
    # We'll skip the complex cross-correlation here for brevity, but it would be 
    # the next logical step to quantify the lag in hours.
else:
    print("Solar irradiance or indoor temperature data not available.")

## 3. PV Heat Weight Bounds Evaluation

Evaluate how the PV heat weight parameter behaves within its defined bounds.

In [ ]:
if 'pv_heat_weight' in df.columns:
    plt.figure(figsize=(12, 5))
    sns.histplot(df['pv_heat_weight'].dropna(), bins=30, kde=True)
    plt.title('Distribution of PV Heat Weight')
    plt.xlabel('PV Heat Weight')
    plt.ylabel('Frequency')
    
    # Assuming bounds are [0.1, 0.9] for this example
    plt.axvline(0.1, color='r', linestyle='--', label='Lower Bound (0.1)')
    plt.axvline(0.9, color='r', linestyle='--', label='Upper Bound (0.9)')
    plt.legend()
    plt.show()
    
    # Time series of PV heat weight
    plt.figure(figsize=(14, 5))
    plt.plot(df.index, df['pv_heat_weight'], color='teal')
    plt.title('PV Heat Weight Over Time')
    plt.xlabel('Time')
    plt.ylabel('Weight')
    plt.axhline(0.1, color='r', linestyle='--', alpha=0.5)
    plt.axhline(0.9, color='r', linestyle='--', alpha=0.5)
    plt.show()
else:
    print("PV heat weight data not available.")

## 4. Multi-Source Heat Contribution

Analyze the relative contributions of primary and secondary heat sources.

In [ ]:
if 'primary_heat_source_power' in df.columns and 'secondary_heat_source_power' in df.columns:
    # Resample to daily for clearer stacked area chart
    daily_power = df[['primary_heat_source_power', 'secondary_heat_source_power']].resample('D').sum()
    
    plt.figure(figsize=(14, 7))
    plt.stackplot(daily_power.index, 
                  daily_power['primary_heat_source_power'], 
                  daily_power['secondary_heat_source_power'],
                  labels=['Primary Source', 'Secondary Source (PV)'],
                  colors=['#ff9999', '#66b3ff'],
                  alpha=0.8)
    
    plt.title('Daily Heat Contribution by Source')
    plt.xlabel('Date')
    plt.ylabel('Total Energy (Arbitrary Units)')
    plt.legend(loc='upper left')
    plt.show()
else:
    print("Heat source power data not available.")